# 1. COVID Policy Data — Download and Cleaning

## Data source

**[COVID-19 Data Hub](https://covid19datahub.io/)** — We use the **Level 1 (national)** dataset.

## Why Level 1 (national)?

Our treatment variable is the **stringency of COVID-19 restrictions**, measured by the
Oxford Government Response Tracker's `stringency_index` (0–100 scale).

We initially investigated using **Level 2 (regional)** data, since in practice many
European countries set pandemic restrictions at the regional level (German Länder,
Italian color zones, Spanish Comunidades, etc.). However, we found that:

1. The pre-computed `stringency_index` at Level 2 has **zero within-country variation**
   for all countries except the UK — all regions within a country receive the same
   national-level score on any given date.
2. The **individual policy indicators** (school closings, workplace closings, etc.)
   also showed no real within-country variation. Our initial check flagged them as
   having variation, but this was driven entirely by the UK; the `.any()` aggregation
   masked the fact that 8 of 9 countries had zero regional variation.
3. The `covid19dh` package stores Level 2 policy values as the **negative of the
   Level 1 values** (i.e., L2 + L1 = 0 exactly), confirming that no real sub-national
   policy data is encoded — the Level 2 policy fields are an artifact, not genuine
   sub-national measurements.

**Therefore, we use Level 1 (national) data.** Our identification strategy relies on
**cross-country variation** in the timing and intensity of lockdowns: different countries
locked down at different times (Italy in early March 2020, Germany mid-March,
Netherlands later) and with different intensities. With city and week fixed effects,
we exploit the fact that at any given time, cities in different countries faced
different levels of restrictions.

**Limitation**: With 9 countries, treatment varies at the country × date level.
Standard errors must be clustered at the country level, and with only 9 clusters,
we use wild cluster bootstrap for valid inference.

## Pipeline overview

1. Download Level 1 COVID data via the `covid19dh` Python package (reproducible)
2. Filter to 9 target EU countries
3. Select policy/stringency and epidemiological variables
4. Compute derived variables (new cases per 100k)
5. Build city-to-country mapping for later merge with air quality data
6. Save cleaned dataset

In [ ]:
import subprocess
import sys

# Install covid19dh if not already installed
# This package provides programmatic access to the COVID-19 Data Hub
# (https://covid19datahub.io/). We use it because the direct CSV download
# URLs return HTTP 403 as of 2026.
try:
    import covid19dh
    print(f'covid19dh already installed (version {covid19dh.__version__})')
except ImportError:
    print('Installing covid19dh...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'covid19dh'])
    print('Done.')

import pandas as pd
import os

## 1.1 Download Level 1 (national) COVID data

The COVID-19 Data Hub provides pre-compiled datasets at three administrative levels.
We use Level 1 (country-level) which contains both the Oxford stringency index and
epidemiological data.

**Download method**: We use the `covid19dh` Python package, which provides a clean
API to the COVID-19 Data Hub. The direct CSV download URL
(`https://storage.covid19datahub.io/level/1.csv.gz`) returns HTTP 403 as of 2026,
so the package is the reliable alternative.

The download step saves the result to `data/raw/covid_level1.csv` so subsequent
runs can load from the local file without re-downloading.

In [ ]:
raw_path = '../data/raw/covid_level1.csv'

TARGET_ISO3 = ['AUT', 'BEL', 'CZE', 'DEU', 'ESP', 'FRA', 'GBR', 'ITA', 'NLD']

if os.path.exists(raw_path):
    print(f'Loading from local file: {raw_path}')
    df_raw = pd.read_csv(raw_path, low_memory=False)
else:
    from covid19dh import covid19

    dfs = []
    for iso in TARGET_ISO3:
        print(f'  Downloading {iso}...', end=' ')
        df_country, _ = covid19(iso, level=1, verbose=False)
        print(f'{len(df_country):,} rows')
        dfs.append(df_country)

    df_raw = pd.concat(dfs, ignore_index=True)
    df_raw.to_csv(raw_path, index=False)
    print(f'Saved to {raw_path}')

print(f'Loaded: {len(df_raw):,} rows × {len(df_raw.columns)} columns')
print(f'Date range: {df_raw["date"].min()} to {df_raw["date"].max()}')
print(f'Countries: {df_raw["iso_alpha_3"].nunique()}')

## 1.2 Filter to 9 target countries

We keep 9 EU countries selected based on:
- **(a)** Good air quality monitor coverage in the AQICN dataset (multiple cities per country)
- **(b)** Diverse lockdown timelines — early movers (Italy: March 9, 2020) vs.
  later movers (Netherlands: March 23, 2020) provide cross-country variation
- **(c)** Geographic spread across Western, Central, and Southern Europe

In [ ]:
TARGET_COUNTRIES = {
    'AUT': 'Austria',
    'BEL': 'Belgium',
    'CZE': 'Czechia',
    'DEU': 'Germany',
    'ESP': 'Spain',
    'FRA': 'France',
    'GBR': 'United Kingdom',
    'ITA': 'Italy',
    'NLD': 'Netherlands'
}

df = df_raw[df_raw['iso_alpha_3'].isin(TARGET_COUNTRIES.keys())].copy()
df['country_name'] = df['iso_alpha_3'].map(TARGET_COUNTRIES)

print(f'Filtered to {df["iso_alpha_3"].nunique()} countries, {len(df):,} rows')

## 1.3 Select and clean variables

We keep:
- **Identifiers**: country name, ISO codes, date
- **Policy variables**: individual Oxford Government Response Tracker (OxCGRT) indicators.
  Each is an ordinal scale capturing restriction intensity. We keep these for
  potential decomposition analysis (which type of restriction matters most for air quality?).
- **Composite index**: `stringency_index` (0–100) — our **primary treatment variable**.
  This is a pre-computed weighted average of closure and containment indicators from
  the Oxford team. Higher values = stricter restrictions.
- **Epidemiological controls**: confirmed cases, deaths, and population — needed to
  control for the pandemic situation itself (policy responds to case counts, creating
  a potential feedback loop we must account for).

In [ ]:
# Check what columns are available
print('All columns:')
print(df.columns.tolist())

In [ ]:
id_cols = ['iso_alpha_2', 'iso_alpha_3', 'country_name', 'date']

# Individual OxCGRT policy indicators (ordinal scales)
# Source: https://github.com/OxCGRT/covid-policy-tracker/blob/master/documentation/codebook.md
policy_cols = [
    'school_closing',                     # 0-3: no measures → require closing all levels
    'workplace_closing',                  # 0-3: no measures → require closing all-but-essential
    'cancel_events',                      # 0-2: no measures → require cancelling
    'gatherings_restrictions',            # 0-4: no restrictions → gatherings <10 people
    'transport_closing',                  # 0-2: no measures → require closing
    'stay_home_restrictions',             # 0-3: no measures → minimal exceptions only
    'internal_movement_restrictions',     # 0-2: no measures → movement restricted
    'international_movement_restrictions' # 0-4: no restrictions → total border closure
]

# Composite indices (pre-computed by Oxford team, 0-100 scale)
index_cols = [
    'stringency_index',            # our primary treatment variable
    'government_response_index',   # broader: includes economic support measures
    'containment_health_index'     # closure + health system indicators
]

# Epidemiological data (for controls)
epi_cols = ['confirmed', 'deaths', 'population']

# Keep only columns that exist in the downloaded data
all_wanted = id_cols + policy_cols + index_cols + epi_cols
keep_cols = [c for c in all_wanted if c in df.columns]
missing = [c for c in all_wanted if c not in df.columns]
if missing:
    print(f'Warning: columns not in data: {missing}')

df = df[keep_cols].copy()
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values(['iso_alpha_3', 'date']).reset_index(drop=True)

print(f'Shape: {df.shape}')
print(f'Date range: {df["date"].min().date()} to {df["date"].max().date()}')
df.head()

## 1.4 Inspect the stringency index

Before proceeding, we check the coverage and basic properties of our treatment variable.
The key thing to verify is that countries have **different** stringency values on the
same dates — this cross-country variation is what drives our identification.

In [ ]:
# Coverage per country
coverage = df.groupby('country_name').agg(
    date_min=('date', 'min'),
    date_max=('date', 'max'),
    n_days=('date', 'count'),
    stringency_pct=('stringency_index', lambda x: x.notna().mean() * 100),
    stringency_mean=('stringency_index', 'mean'),
    stringency_max=('stringency_index', 'max')
).round(1)

print('Stringency index coverage per country:')
print(coverage.to_string())

In [ ]:
# Cross-country variation over time — this is what drives our identification
# We show stringency values at key dates during the first wave to illustrate
# the staggered lockdown timing across countries
key_dates = ['2020-03-01', '2020-03-15', '2020-03-22', '2020-04-01', '2020-05-15']
for d in key_dates:
    row = df[df['date'] == d].set_index('country_name')['stringency_index']
    if len(row) > 0:
        print(f'\n{d}:')
        print(row.sort_values(ascending=False).to_string())

## 1.5 Compute derived variables

We compute:
- **Daily new cases** from cumulative `confirmed`, and a **7-day rolling average
  per 100k population**. These serve as control variables — policy stringency partly
  responds to case counts, so controlling for the epidemiological situation helps
  isolate the direct effect of policy on air quality.
- **Daily new deaths** (same logic).

Negative diffs (from data corrections) are clipped to 0.

In [ ]:
# Confirmed cases are cumulative → compute daily new cases
df['new_cases'] = df.groupby('iso_alpha_3')['confirmed'].diff().clip(lower=0)
df['new_deaths'] = df.groupby('iso_alpha_3')['deaths'].diff().clip(lower=0)

# 7-day rolling average of new cases per 100k population
df['cases_7d_avg'] = (
    df.groupby('iso_alpha_3')['new_cases']
    .transform(lambda x: x.rolling(7, min_periods=1).mean())
)
df['cases_per_100k'] = df['cases_7d_avg'] / df['population'] * 100_000

print('Derived variables added.')
df[['country_name', 'date', 'stringency_index', 'new_cases', 'cases_per_100k']].dropna().head(10)

## 1.6 City-to-country mapping

Our air quality data (AQICN) uses **city names** with 2-letter country codes (ISO 3166-1
alpha-2), while the COVID data uses 3-letter ISO codes (alpha-3). We define the full
list of 126 AQICN cities and their country codes so that the merge notebook can join
city-level pollution data with national-level policy data.

The mapping is saved as a CSV for reuse across notebooks.

In [ ]:
# 2-letter (AQICN) to 3-letter (COVID hub) country code mapping
ISO2_TO_ISO3 = {
    'AT': 'AUT', 'BE': 'BEL', 'CZ': 'CZE', 'DE': 'DEU', 'ES': 'ESP',
    'FR': 'FRA', 'GB': 'GBR', 'IT': 'ITA', 'NL': 'NLD'
}

# Full list of AQICN cities per country
# These were identified from the AQICN COVID-19 data platform by filtering
# to our 9 target countries and keeping cities with >= 60 days of NO2 data
# in the Q2 2020 sample period.
city_country = [
    # Austria (5 cities)
    ('AT', 'Graz'), ('AT', 'Innsbruck'), ('AT', 'Linz'),
    ('AT', 'Salzburg'), ('AT', 'Vienna'),

    # Belgium (6 cities)
    ('BE', 'Antwerpen'), ('BE', 'Brussels'), ('BE', 'Charleroi'),
    ('BE', 'Gent'), ('BE', 'Liège'), ('BE', 'Namur'),

    # Czechia (5 cities)
    ('CZ', 'Brno'), ('CZ', 'Olomouc'), ('CZ', 'Ostrava'),
    ('CZ', 'Pilsen'), ('CZ', 'Prague'),

    # Germany (17 cities)
    ('DE', 'Augsburg'), ('DE', 'Berlin'), ('DE', 'Darmstadt'),
    ('DE', 'Dresden'), ('DE', 'Düsseldorf'), ('DE', 'Freiburg'),
    ('DE', 'Hamburg'), ('DE', 'Hannover'), ('DE', 'Karlsruhe'),
    ('DE', 'Kassel'), ('DE', 'Köln'), ('DE', 'Mainz'),
    ('DE', 'Munich'), ('DE', 'Münster'), ('DE', 'Potsdam'),
    ('DE', 'Stuttgart'), ('DE', 'Wiesbaden'),

    # Spain (23 cities)
    ('ES', 'Barcelona'), ('ES', 'Bilbao'), ('ES', 'Burgos'),
    ('ES', 'Castelló de la Plana'), ('ES', 'Córdoba'),
    ('ES', 'Donostia / San Sebastián'), ('ES', 'Gasteiz / Vitoria'),
    ('ES', 'Granada'), ('ES', 'Huelva'),
    ('ES', 'Las Palmas de Gran Canaria'), ('ES', 'Madrid'),
    ('ES', 'Murcia'), ('ES', 'Málaga'), ('ES', 'Oviedo'),
    ('ES', 'Palma'), ('ES', 'Pamplona'), ('ES', 'Salamanca'),
    ('ES', 'Santa Cruz de Tenerife'), ('ES', 'Santander'),
    ('ES', 'Sevilla'), ('ES', 'Valencia'), ('ES', 'Valladolid'),
    ('ES', 'Zaragoza'),

    # France (27 cities)
    ('FR', 'Amiens'), ('FR', 'Besançon'), ('FR', 'Bordeaux'),
    ('FR', 'Caen'), ('FR', 'Clermont-Ferrand'), ('FR', 'Dijon'),
    ('FR', 'Grenoble'), ('FR', 'Lille'), ('FR', 'Limoges'),
    ('FR', 'Lyon'), ('FR', 'Marseille'), ('FR', 'Metz'),
    ('FR', 'Montpellier'), ('FR', 'Nancy'), ('FR', 'Nantes'),
    ('FR', 'Nice'), ('FR', 'Nîmes'), ('FR', 'Orléans'),
    ('FR', 'Paris'), ('FR', 'Perpignan'), ('FR', 'Rennes'),
    ('FR', 'Rouen'), ('FR', 'Saint-Étienne'), ('FR', 'Strasbourg'),
    ('FR', 'Toulon'), ('FR', 'Toulouse'), ('FR', 'Tours'),

    # United Kingdom (20 cities)
    ('GB', 'Belfast'), ('GB', 'Birmingham'), ('GB', 'Bristol'),
    ('GB', 'Cardiff'), ('GB', 'Coventry'), ('GB', 'Edinburgh'),
    ('GB', 'Glasgow'), ('GB', 'Leeds'), ('GB', 'Leicester'),
    ('GB', 'Liverpool'), ('GB', 'London'), ('GB', 'Manchester'),
    ('GB', 'Newport'), ('GB', 'Norwich'), ('GB', 'Plymouth'),
    ('GB', 'Preston'), ('GB', 'Reading'), ('GB', 'Sheffield'),
    ('GB', 'Southend-on-Sea'), ('GB', 'Swansea'),

    # Italy (12 cities)
    ('IT', 'Bologna'), ('IT', 'Brescia'), ('IT', 'Florence'),
    ('IT', 'Livorno'), ('IT', 'Milan'), ('IT', 'Modena'),
    ('IT', 'Naples'), ('IT', 'Parma'), ('IT', 'Prato'),
    ('IT', 'Rome'), ('IT', 'Trieste'), ('IT', 'Turin'),

    # Netherlands (11 cities)
    ('NL', 'Amsterdam'), ('NL', 'Breda'), ('NL', 'Dordrecht'),
    ('NL', 'Eindhoven'), ('NL', 'Groningen'), ('NL', 'Haarlem'),
    ('NL', 'Maastricht'), ('NL', 'Nijmegen'), ('NL', 'Rotterdam'),
    ('NL', 'The Hague'), ('NL', 'Utrecht'),
]

df_map = pd.DataFrame(city_country, columns=['aqicn_country', 'aqicn_city'])
df_map['iso_alpha_3'] = df_map['aqicn_country'].map(ISO2_TO_ISO3)

print(f'Total cities: {len(df_map)}')
print(f'Countries: {df_map["iso_alpha_3"].nunique()}')
print(f'\nCities per country:')
print(df_map.groupby('iso_alpha_3')['aqicn_city'].count().to_string())

## 1.7 Save outputs

In [ ]:
# Save cleaned COVID policy data (national level)
output_path = '../data/clean/covid_policy_national.csv'
df.to_csv(output_path, index=False)
print(f'Saved: {output_path} ({len(df):,} rows)')

# Save city-to-country mapping
map_path = '../data/clean/city_country_mapping.csv'
df_map.to_csv(map_path, index=False)
print(f'Saved: {map_path} ({len(df_map)} cities)')